# DeepSeek OCR Testing
This notebook tests the local `deepseek-ocr` model. It will:
1. Display an input image.
2. Send the image to the local Ollama server running `deepseek-ocr`.
3. Render the extracted output (Text and Tables) as Markdown.

### Expected Input/Output:
* **Input**: An image containing text and/or tables.
* **Output**: The extracted Text as well as correctly formatted Markdown Tables.

In [ ]:
!pip install pillow requests IPython

In [ ]:
import os
import base64
import requests
from PIL import Image
from IPython.display import display, Markdown, Image as IPImage

# Local Ollama connection where your AMD server is linked
OLLAMA_SERVER_URL = "http://localhost:11434"
OLLAMA_VISION_MODEL = "deepseek-ocr"

print(f"Testing Model: {OLLAMA_VISION_MODEL}")

In [1]:
# 1. Provide an Image Path
# Replace this with the path to the image you want to test (e.g., .png or .jpg).
# It can contain text, tables, or generic information.
test_image_path = "test_image.png" # <--- Change this to your image file

if not os.path.exists(test_image_path):
    print(f"Please place an image at {test_image_path} to test.")
    # Creating a dummy file so the script runs without errors visually
    # but the image view will naturally fail until replaced.
else:
    print("Input Image:")
    display(IPImage(filename=test_image_path, width=400))

NameError: name 'os' is not defined

In [ ]:
# 2. Function to request DeepSeek OCR
def analyze_image_with_ocr(image_path):
    endpoint = f"{OLLAMA_SERVER_URL}/api/generate"
    
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
        
    payload = {
        "model": OLLAMA_VISION_MODEL,
        # A prompt asking it to specifically extract text and format tables.
        "prompt": "Extract all text and tabular data from this image perfectly. If there is a table, format it explicitly as a Markdown table.",
        "images": [img_b64],
        "stream": False
    }
    
    try:
        print("Sending request to DeepSeek OCR (this may take a moment)...")
        resp = requests.post(endpoint, json=payload)
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        return f"Error connecting to Ollama: {e}" 

In [ ]:
# 3. Analyze and Output Text & Tables
if os.path.exists(test_image_path):
    # This will generate both text and a markdown table if there is table data
    result_markdown = analyze_image_with_ocr(test_image_path)
    
    print("================ RAW OCR OUTPUT ================\n")
    # Displaying it as raw text so we can see what the model actually outputs
    print(result_markdown)
    
    print("\n================ RENDERED JUPYTER OUTPUT ================\n")
    # Displaying it as Markdown so tables look correctly structured in Jupyter
    display(Markdown(result_markdown))